# Benchmark FAISS — Title Embeddings

So sánh 3 loại FAISS index trên `title_embeddings.npy` (~1.3M vectors, dim=384)  
để chọn cấu hình tối ưu cho production.



## 1. Imports & cấu hình

In [1]:
import os, tempfile, time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"D:\School\bigdata\BigData_Nhom_6")

EMB_PATH = PROJECT_ROOT / "data/faiss/title_embeddings.npy"

SAMPLE_SIZE = 100_000
N_QUERIES   = 200
TOPK        = 10
SEED        = 42
MIN_RECALL  = 0.97

IVF_NLIST      = 1024
IVF_TRAIN_SIZE = 50_000
IVF_NPROBES    = [4, 8, 16, 32, 64]

HNSW_M               = 32
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCHES     = [16, 32, 64, 128]

## 2. Load & chuẩn hoá dữ liệu


In [2]:
x_all = np.load(EMB_PATH, mmap_mode="r")
print("embedding shape:", x_all.shape)
print("dtype:", x_all.dtype)

xb = np.ascontiguousarray(x_all[:min(SAMPLE_SIZE, x_all.shape[0])].astype("float32"))
faiss.normalize_L2(xb)
n, d = xb.shape

rng = np.random.default_rng(SEED)
xq  = np.ascontiguousarray(xb[rng.choice(n, size=min(N_QUERIES, n), replace=False)].copy())

print(f"xb: {xb.shape}  xq: {xq.shape}")

embedding shape: (1280299, 384)
dtype: float32
xb: (100000, 384)  xq: (200, 384)


## 3. Benchmark — FlatIP / IVFFlat / HNSWFlat

In [ ]:
def get_size_mb(index):
    f = tempfile.NamedTemporaryFile(delete=False, suffix=".index")
    f.close()
    faiss.write_index(index, f.name)
    size = os.path.getsize(f.name) / 1024 / 1024
    os.remove(f.name)
    return round(size, 2)


def recall_at_k(pred, gt, k):
    recalls = []
    for p, g in zip(pred[:, :k], gt[:, :k]):
        matched = 0
        for pid in p:
            if pid in g:
                matched += 1
            else:
                # near-duplicate check: xb đã normalized → IP = cosine
                sim = xb[pid] @ xb[g].T
                if sim.max() >= 0.9999:
                    matched += 1
        recalls.append(matched / k)
    return float(np.mean(recalls))


def run_search(index, name, gt_ids):
    index.search(xq[:5], TOPK)  # warmup
    t0 = time.perf_counter()
    _, ids = index.search(xq, TOPK)
    sec = time.perf_counter() - t0
    return {
        "index_name":     name,
        "build_sec":      None,
        "size_mb":        None,
        "search_sec":     round(sec, 4),
        "ms_per_query":   round(sec * 1000 / len(xq), 4),
        "qps":            round(len(xq) / sec, 2),
        f"recall@{TOPK}": round(recall_at_k(ids, gt_ids, TOPK), 4),
    }

In [4]:
results = []

# FlatIP — ground truth
t0 = time.perf_counter()
flat = faiss.IndexFlatIP(d)
flat.add(xb)
build_sec = time.perf_counter() - t0

_, gt_ids = flat.search(xq, TOPK)

row = run_search(flat, "FlatIP_exact", gt_ids)
row["build_sec"] = round(build_sec, 4)
row["size_mb"]   = get_size_mb(flat)
results.append(row)
print("FlatIP done")

# IVFFlat
quantizer = faiss.IndexFlatIP(d)
ivf = faiss.IndexIVFFlat(quantizer, d, IVF_NLIST, faiss.METRIC_INNER_PRODUCT)

train_ids = np.random.default_rng(SEED + 1).choice(n, size=min(IVF_TRAIN_SIZE, n), replace=False)

t0 = time.perf_counter()
ivf.train(xb[train_ids])
ivf.add(xb)
build_sec = time.perf_counter() - t0
size_mb   = get_size_mb(ivf)

for nprobe in IVF_NPROBES:
    ivf.nprobe = nprobe
    row = run_search(ivf, f"IVFFlat_nprobe={nprobe}", gt_ids)
    row["build_sec"] = round(build_sec, 4)
    row["size_mb"]   = size_mb
    results.append(row)
print("IVFFlat done")

# HNSWFlat
hnsw = faiss.IndexHNSWFlat(d, HNSW_M, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = HNSW_EF_CONSTRUCTION

t0 = time.perf_counter()
hnsw.add(xb)
build_sec = time.perf_counter() - t0
size_mb   = get_size_mb(hnsw)

for ef in HNSW_EF_SEARCHES:
    hnsw.hnsw.efSearch = ef
    row = run_search(hnsw, f"HNSWFlat_ef={ef}", gt_ids)
    row["build_sec"] = round(build_sec, 4)
    row["size_mb"]   = size_mb
    results.append(row)
print("HNSW done")

FlatIP done
IVFFlat done
HNSW done


In [5]:
cols = ["index_name", "build_sec", "size_mb", "search_sec", "ms_per_query", "qps", f"recall@{TOPK}"]
df   = pd.DataFrame(results)[cols]

base_ms = df.loc[df["index_name"] == "FlatIP_exact", "ms_per_query"].iloc[0]
df["speedup_vs_flat"] = (base_ms / df["ms_per_query"]).round(2)

display(df)

# chọn config — loại FlatIP khỏi ứng viên
candidates = df[df["index_name"] != "FlatIP_exact"]
good = candidates[candidates[f"recall@{TOPK}"] >= MIN_RECALL]

if len(good):
    best = good.sort_values("ms_per_query").iloc[0]
else:
    best = candidates.sort_values([f"recall@{TOPK}", "ms_per_query"], ascending=[False, True]).iloc[0]
    print(f"[!] không có config nào recall >= {MIN_RECALL}, chọn recall cao nhất")

print(f"\nbest config : {best['index_name']}")
print(f"recall@{TOPK}  : {best[f'recall@{TOPK}']}")
print(f"ms/query    : {best['ms_per_query']}")
print(f"speedup     : {best['speedup_vs_flat']}x")

,index_name,build_sec,size_mb,search_sec,ms_per_query,qps,recall@10,speedup_vs_flat
0,FlatIP_exact,0.1031,146.48,0.1797,0.8983,1113.23,1.0000,1.00
1,IVFFlat_nprobe=4,5.7536,148.76,0.0242,0.1211,8255.29,0.9465,7.42
2,IVFFlat_nprobe=8,5.7536,148.76,0.0532,0.2658,3762.64,0.9730,3.38
3,IVFFlat_nprobe=16,5.7536,148.76,0.0791,0.3954,2529.08,0.9790,2.27
4,IVFFlat_nprobe=32,5.7536,148.76,0.0997,0.4986,2005.72,0.9875,1.80
5,IVFFlat_nprobe=64,5.7536,148.76,0.1369,0.6847,1460.46,0.9915,1.31
6,HNSWFlat_ef=16,70.1114,172.44,0.0070,0.0349,28640.16,0.6845,25.74
7,HNSWFlat_ef=32,70.1114,172.44,0.0103,0.0516,19363.71,0.6900,17.41
8,HNSWFlat_ef=64,70.1114,172.44,0.0139,0.0694,14406.83,0.6950,12.94
9,HNSWFlat_ef=128,70.1114,172.44,0.0316,0.1580,6330.56,0.7055,5.69



best config : IVFFlat_nprobe=8
recall@10  : 0.973
ms/query    : 0.2658
speedup     : 3.38x
